# OpenFold3 ddG Stand

Ноутбук для запуска panel-based ddG стенда: WT плюс 19 замен для каждой выбранной позиции.

Теперь доступны два режима ввода:
- ручной список `molecules`
- прямой ввод `PDB ID`, когда notebook сам скачивает `.cif`, извлекает белковые цепи и строит WT-запрос

Для позиций доступны три режима:
- `explicit_list`: явный список и диапазоны, например `10-25, 40, 50-60`
- `all_chain_positions`: взять все позиции выбранной цепи
- в обоих случаях на каждой позиции будут построены все 19 замен, кроме WT-остатка


In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_ROOT.parent
HELPERS_DIR = NOTEBOOK_ROOT / 'helpers'
OPENFOLD_REPO_DIR = Path(os.environ.get('OPENFOLD_REPO_DIR', str(REPO_ROOT / 'openfold-3'))).resolve()
SAFE_TRITON_CACHE_DIR = Path(os.environ.get('TRITON_CACHE_DIR', '/tmp/openfold3_triton_cache')).resolve()

os.environ.setdefault('OPENFOLD_DISABLE_OPTIONAL_DEEPSPEED_IMPORT', '1')
os.environ.setdefault('TRITON_CACHE_DIR', str(SAFE_TRITON_CACHE_DIR))
os.environ.setdefault('DS_BUILD_AIO', '0')
os.environ.setdefault('AIO', '0')
SAFE_TRITON_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for path in (NOTEBOOK_ROOT, HELPERS_DIR, OPENFOLD_REPO_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from of_notebook_lib import RuntimeConfig, preview_molecules, validate_molecules
from of_notebook_lib.query_builders import build_single_query_payload, normalize_molecules
from openfold3.panel_stand import PanelDdgStandRunner, PanelStandConfig
from openfold3.panel_summary import summarize_panel_state_db, write_panel_summary_outputs
from openfold3_length_benchmark.composition import collect_entry_compositions, normalize_pdb_id

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)
CANONICAL_AA = 'ACDEFGHIKLMNPQRSTVWY'

def parse_positions_spec(positions_text: str, *, sequence_length: int) -> tuple[int, ...]:
    values: list[int] = []
    for token in positions_text.replace('\n', ',').split(','):
        token = token.strip()
        if not token:
            continue
        if "-" in token:
            start_raw, end_raw = token.split("-", 1)
            start = int(start_raw.strip())
            end = int(end_raw.strip())
            if start > end:
                raise ValueError(f"Некорректный диапазон: {token}")
            values.extend(range(start, end + 1))
            continue
        values.append(int(token))
    if not values:
        raise ValueError("Нужно указать хотя бы одну позицию")
    unique_values = tuple(sorted(set(values)))
    for value in unique_values:
        if value < 1 or value > sequence_length:
            raise ValueError(f"Позиция {value} выходит за длину цепи ({sequence_length})")
    return unique_values

def resolve_positions(*, positions_mode: str, positions_text: str, sequence_length: int) -> tuple[int, ...]:
    if positions_mode == "all_chain_positions":
        return tuple(range(1, sequence_length + 1))
    if positions_mode != "explicit_list":
        raise ValueError(f"Неизвестный positions_mode: {positions_mode}")
    return parse_positions_spec(positions_text, sequence_length=sequence_length)

def find_chain_sequence(molecules: list[dict], mutable_chain_id: str) -> str:
    for molecule in normalize_molecules(molecules):
        if mutable_chain_id in molecule["chain_ids"]:
            sequence = molecule.get("sequence")
            if not sequence:
                raise ValueError(f"У цепи {mutable_chain_id} нет sequence")
            return str(sequence)
    raise ValueError(f"Цепь {mutable_chain_id} не найдена во входных молекулах")

def build_panel_preview(target_id: str, molecules: list[dict], mutable_chain_id: str, positions: tuple[int, ...]) -> pd.DataFrame:
    sequence = find_chain_sequence(molecules, mutable_chain_id)
    rows = []
    for position in positions:
        wt_residue = sequence[position - 1]
        rows.append({
            "panel_id": f"{target_id}_{mutable_chain_id}_{wt_residue}{position}",
            "цепь": mutable_chain_id,
            "позиция": position,
            "WT остаток": wt_residue,
            "число мутантов": len(CANONICAL_AA) - 1,
        })
    return pd.DataFrame(rows)

def render_info_card(title: str, rows: list[tuple[str, object]], accent: str = "#0b7285") -> None:
    items = "".join(
        f"<tr><td style='padding:6px 10px;font-weight:600;white-space:nowrap;vertical-align:top;'>{label}</td><td style='padding:6px 10px;'>{value}</td></tr>"
        for label, value in rows
    )
    html = f"""
    <div style="border:1px solid #d0d7de;border-left:6px solid {accent};border-radius:10px;padding:12px 14px;margin:8px 0;background:#fbfdff;">
      <div style="font-size:18px;font-weight:700;margin-bottom:8px;">{title}</div>
      <table style="border-collapse:collapse;">{items}</table>
    </div>
    """
    display(HTML(html))

def resolve_experiment_molecules(*, input_mode: str, molecules: list[dict], pdb_id: str | None, pdb_cache_dir: str | Path | None = None) -> tuple[list[dict], pd.DataFrame | None, dict[str, object]]:
    if input_mode == "manual_molecules":
        normalized = normalize_molecules(molecules)
        return normalized, None, {"input_mode": input_mode}
    if input_mode != "pdb_id":
        raise ValueError(f"Неизвестный input_mode: {input_mode}")
    if not pdb_id:
        raise ValueError("Для input_mode='pdb_id' нужно указать pdb_id")
    normalized_pdb_id = normalize_pdb_id(pdb_id)
    composition = collect_entry_compositions(normalized_pdb_id, cache_dir=pdb_cache_dir, max_entries=1)[0]
    preview_df = pd.DataFrame([composition.to_preview_row()])
    if composition.status != "ok":
        raise ValueError(f"Не удалось разобрать PDB {normalized_pdb_id}: {composition.issue}")
    metadata = {
        "input_mode": input_mode,
        "pdb_id": composition.pdb_id,
        "reference_path": None if composition.source_path is None else str(composition.source_path),
        "chain_ids": ",".join(composition.chain_ids),
        "total_protein_length": composition.total_protein_length,
    }
    return composition.molecules, preview_df, metadata


## 1. Настройка runtime


In [ ]:
runtime = RuntimeConfig(
    project_dir=REPO_ROOT,
    openfold_repo_dir=OPENFOLD_REPO_DIR,
    openfold_prefix=Path(os.environ.get("OPENFOLD_PREFIX", str(REPO_ROOT / ".venv"))),
    results_dir=REPO_ROOT / "results",
    msa_cache_dir=REPO_ROOT / "msa_cache" / "colabfold_msas",
    triton_cache_dir=SAFE_TRITON_CACHE_DIR,
    fixed_msa_tmp_dir=REPO_ROOT / ".runtime" / "of3_colabfold_msas",
    use_fused_attention=False,
    use_deepspeed=False,
)

runtime_env = runtime.build_env()
for key, value in runtime_env.items():
    os.environ[key] = value
os.environ["OPENFOLD_PROJECT_DIR"] = str(runtime.project_dir)
os.environ["OPENFOLD_REPO_DIR"] = str(runtime.openfold_repo_dir)

render_info_card("Runtime и безопасные env-переменные", [("project_dir", runtime.project_dir), ("openfold_repo_dir", runtime.openfold_repo_dir), ("openfold_runner", runtime.openfold_runner), ("TRITON_CACHE_DIR", os.environ.get("TRITON_CACHE_DIR")), ("OPENFOLD_DISABLE_OPTIONAL_DEEPSPEED_IMPORT", os.environ.get("OPENFOLD_DISABLE_OPTIONAL_DEEPSPEED_IMPORT")), ("DS_BUILD_AIO", os.environ.get("DS_BUILD_AIO"))], accent="#1c7ed6")


## 2. Диагностика окружения


In [ ]:
deepspeed_status = "не установлен"
deepspeed_hint = "Для ddG stand это допустимо: notebook по умолчанию работает без optional DeepSpeed import."
try:
    import importlib.util
    if importlib.util.find_spec("deepspeed") is not None:
        deepspeed_status = "установлен"
        deepspeed_hint = "Если захочешь включать DeepSpeed-ускорение, на сервере должен быть установлен libaio-dev/libaio1-dev."
except Exception as exc:
    deepspeed_status = f"ошибка проверки: {exc}"
libaio_candidates = [Path("/usr/lib/x86_64-linux-gnu/libaio.so"), Path("/usr/lib64/libaio.so"), Path("/lib/x86_64-linux-gnu/libaio.so.1")]
libaio_found = any(path.exists() for path in libaio_candidates)
runner_found = Path(runtime.openfold_runner).exists()
render_info_card("Проверка окружения", [("run_openfold найден", runner_found), ("DeepSpeed", deepspeed_status), ("libaio найден", libaio_found), ("Подсказка", deepspeed_hint)], accent="#2b8a3e" if runner_found else "#c92a2a")


## 3. Основная настройка эксперимента

Можно выбрать ручной ввод молекул или загрузку по PDB ID, а также явный список позиций или все позиции цепи.


In [ ]:
preset_name = "balanced"
presets = {
    "fast_smoke": {"num_diffusion_samples": 1, "num_model_seeds": 1, "msa_panel_workers": 1, "analysis_workers": 2, "cleanup_intermediates": True},
    "balanced": {"num_diffusion_samples": 2, "num_model_seeds": 1, "msa_panel_workers": 1, "analysis_workers": 4, "cleanup_intermediates": True},
    "thorough": {"num_diffusion_samples": 4, "num_model_seeds": 2, "msa_panel_workers": 2, "analysis_workers": 6, "cleanup_intermediates": False},
}

preset = presets[preset_name]

input_mode = "manual_molecules"
# input_mode = "pdb_id"

positions_mode = "explicit_list"
# positions_mode = "all_chain_positions"

experiment_name = "ddg_panel_demo"
target_id = "demo_target"

pdb_id = "1UBQ"
pdb_cache_dir = NOTEBOOK_ROOT / ".pdb_cache"

molecules = [
    {"molecule_type": "protein", "chain_ids": ["A"], "sequence": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"},
    {"molecule_type": "protein", "chain_ids": ["B"], "sequence": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"},
]

mutable_chain_id = "B"
positions_text = "10-15, 20, 25-27"

runner_yaml = None
inference_ckpt_path = None
inference_ckpt_name = None
msa_computation_settings_yaml = None

num_diffusion_samples = preset["num_diffusion_samples"]
num_model_seeds = preset["num_model_seeds"]
msa_panel_workers = preset["msa_panel_workers"]
analysis_workers = preset["analysis_workers"]
cleanup_intermediates = preset["cleanup_intermediates"]
enable_profiling = True
profiling_sample_interval_seconds = 1.0

output_parent = runtime.results_dir / "panel_ddg_stand"
summary_export_dirname = "summary_exports"


## 4. Предпросмотр входа

В режиме `pdb_id` notebook сам скачивает запись, извлекает белковые цепи и показывает состав. В режиме `all_chain_positions` автоматически будут использованы все позиции выбранной цепи.


In [ ]:
resolved_molecules, pdb_preview_df, input_metadata = resolve_experiment_molecules(input_mode=input_mode, molecules=molecules, pdb_id=pdb_id, pdb_cache_dir=pdb_cache_dir)

issues = validate_molecules(resolved_molecules)
if issues:
    for issue in issues:
        print("INPUT ISSUE:", issue)
else:
    print("Входные молекулы выглядят корректно.")

sequence = find_chain_sequence(resolved_molecules, mutable_chain_id)
positions = resolve_positions(positions_mode=positions_mode, positions_text=positions_text, sequence_length=len(sequence))
panel_preview_df = build_panel_preview(target_id, resolved_molecules, mutable_chain_id, positions)
planned_jobs = int(panel_preview_df["число мутантов"].sum())

render_info_card("План запуска", [("input_mode", input_mode), ("positions_mode", positions_mode), ("target_id", target_id), ("mutable_chain_id", mutable_chain_id), ("длина mutable chain", len(sequence)), ("позиции", list(positions)[:20] if len(positions) > 20 else list(positions)), ("число выбранных позиций", len(positions)), ("число панелей", len(panel_preview_df)), ("число мутантных задач", planned_jobs), ("preset", preset_name), *[(key, value) for key, value in input_metadata.items()]], accent="#7048e8")

display(preview_molecules(resolved_molecules))
if pdb_preview_df is not None:
    display(pdb_preview_df)
display(panel_preview_df.head(50))


## 5. Итоговый план запуска


In [ ]:
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_root = output_parent / f"{experiment_name}_{run_stamp}"
wt_query_path = run_root / "wt_query.json"
wt_query_payload = build_single_query_payload(f"{target_id}_WT", resolved_molecules)

resolved_plan = {
    "input_mode": input_mode,
    "positions_mode": positions_mode,
    "run_root": str(run_root),
    "wt_query_path": str(wt_query_path),
    "target_id": target_id,
    "mutable_chain_id": mutable_chain_id,
    "positions_count": len(positions),
    "positions_preview": list(positions[:20]),
    "runner_yaml": None if runner_yaml is None else str(Path(runner_yaml)),
    "inference_ckpt_path": None if inference_ckpt_path is None else str(Path(inference_ckpt_path)),
    "inference_ckpt_name": inference_ckpt_name,
    "msa_computation_settings_yaml": None if msa_computation_settings_yaml is None else str(Path(msa_computation_settings_yaml)),
    "num_diffusion_samples": num_diffusion_samples,
    "num_model_seeds": num_model_seeds,
    "msa_panel_workers": msa_panel_workers,
    "analysis_workers": analysis_workers,
    "cleanup_intermediates": cleanup_intermediates,
    "enable_profiling": enable_profiling,
    "profiling_sample_interval_seconds": profiling_sample_interval_seconds,
    **{key: value for key, value in input_metadata.items()},
}

render_info_card("Собранный план", [(key, value) for key, value in resolved_plan.items()], accent="#e67700")


## 6. Запуск эксперимента

Backend теперь использует один общий batched predict-worker для всех готовых панелей, чтобы не грузить checkpoint заново на каждой позиции.


CPU/GPU-профилирование включается отдельным профайлером и пишет артефакты в `run_root/profiling/`.

In [ ]:
run_root.mkdir(parents=True, exist_ok=True)
wt_query_path.write_text(json.dumps(wt_query_payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

panel_config = PanelStandConfig(target_id=target_id, wt_query_json=wt_query_path, output_root=run_root, mutable_chain_id=mutable_chain_id, positions=positions, runner_yaml=None if runner_yaml is None else Path(runner_yaml), inference_ckpt_path=None if inference_ckpt_path is None else Path(inference_ckpt_path), inference_ckpt_name=inference_ckpt_name, msa_computation_settings_yaml=None if msa_computation_settings_yaml is None else Path(msa_computation_settings_yaml), num_diffusion_samples=num_diffusion_samples, num_model_seeds=num_model_seeds, msa_panel_workers=msa_panel_workers, analysis_workers=analysis_workers, cleanup_intermediates=cleanup_intermediates, enable_profiling=enable_profiling, profiling_sample_interval_seconds=profiling_sample_interval_seconds)

runner = PanelDdgStandRunner(panel_config)
try:
    run_payload = runner.run()
finally:
    runner.close()

panel_summary = summarize_panel_state_db(Path(run_payload["state_db"]))
summary_outputs = write_panel_summary_outputs(panel_summary, Path(run_payload["output_root"]) / summary_export_dirname)

profiling_summary = None
if run_payload.get("profiling_summary_path"):
    profiling_summary = json.loads(Path(run_payload["profiling_summary_path"]).read_text(encoding="utf-8"))

panel_rows_df = pd.DataFrame([{
    "mutation": f"{row.chain_id}:{row.from_residue}{row.position_1based}{row.to_residue}",
    "predict_status": row.predict_status,
    "analysis_status": row.analysis_status,
    "consensus_z": row.consensus_z,
    "consensus_rank_desc": row.consensus_rank_desc,
    "rosetta_delta_vs_wt": row.rosetta_delta_vs_wt,
    **{f"score__{method}": row.method_scores.get(method) for method in panel_summary.methods},
} for row in panel_summary.rows])
top_consensus_df = pd.DataFrame(panel_summary.top_consensus)

render_info_card("Эксперимент завершён", [("run_root", run_payload["output_root"]), ("state_db", run_payload["state_db"]), ("summary_path", run_payload["summary_path"]), ("panel_count", run_payload["panel_count"]), ("mutant_job_count", run_payload["mutant_job_count"]), ("profiling_summary_path", run_payload.get("profiling_summary_path")), ("profiling_events_path", run_payload.get("profiling_events_path")), ("profiling_samples_path", run_payload.get("profiling_samples_path")), ("profiling_timeline_svg_path", run_payload.get("profiling_timeline_svg_path"))], accent="#2f9e44")
print(json.dumps({key: str(value) for key, value in summary_outputs.items()}, indent=2))

if profiling_summary is not None:
    render_info_card("Профилирование CPU/GPU", [("wall_seconds", round(float(profiling_summary.get("wall_seconds", 0.0)), 2)), ("checkpoint_load_seconds", profiling_summary.get("checkpoint_load_seconds")), ("predict_seconds", profiling_summary.get("predict_seconds")), ("peak_cpu_percent", profiling_summary.get("peak_cpu_percent")), ("peak_rss_gb", profiling_summary.get("peak_rss_gb")), ("max_gpu_util_percent", profiling_summary.get("max_gpu_util_percent")), ("peak_gpu_memory_gb", profiling_summary.get("peak_gpu_memory_gb")), ("process_count_peak", profiling_summary.get("process_count_peak"))], accent="#0b7285")


## 7. Результаты


In [ ]:
render_info_card("Итоговые метрики", [("target_id", panel_summary.target_id), ("total_jobs", panel_summary.total_jobs), ("analyzed_jobs", panel_summary.analyzed_jobs), ("fully_scored_jobs", panel_summary.fully_scored_jobs), ("methods", ", ".join(panel_summary.methods))], accent="#5f3dc4")

display(top_consensus_df)
display(panel_rows_df.sort_values(["consensus_rank_desc", "mutation"], na_position="last"))


## 8. Повторная сборка summary без нового запуска


In [ ]:
existing_run_root = None
# existing_run_root = run_root

if existing_run_root:
    existing_run_root = Path(existing_run_root)
    existing_summary = summarize_panel_state_db(existing_run_root / "state.sqlite")
    existing_outputs = write_panel_summary_outputs(existing_summary, existing_run_root / summary_export_dirname)
    render_info_card("Summary rebuilt", [(key, value) for key, value in {k: str(v) for k, v in existing_outputs.items()}.items()], accent="#0b7285")
    display(pd.DataFrame(existing_summary.top_consensus))
else:
    print("Если нужно просто пересобрать summary, укажи existing_run_root на старую директорию запуска.")
